# ZelusBench: Selective Attention

**Does the model get distracted by irrelevant but salient information?**

This task measures selective attention by injecting varying amounts of distractor
points and relationships into the scenario. The distractors are either disconnected
subgraphs, irrelevant branches, or restatements of known info.

Distractor levels:
- **Clean** (0:1): no distractors
- **Low** (1:1): equal irrelevant to relevant points
- **High** (3:1): 3x as many irrelevant points
- **Extreme** (10:1): 10x irrelevant points — heavy noise

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import os

os.environ["RENDER_SUBRUNS"] = "False"

In [ ]:
from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd)) for qd, rp in zip(query_dicts, parts)]

In [ ]:
DISTRACTOR_LEVELS = {
    0: DistractorLevel.CLEAN,
    1: DistractorLevel.LOW,
    3: DistractorLevel.HIGH,
    10: DistractorLevel.EXTREME,
}

@kbench.task(store_task=False)
def selective_scenario(llm, distractor_ratio: int, seed: int) -> float:
    """Run one scenario with a given distractor density."""
    cfg = ScenarioConfig(
        dim=3,
        min_chain_depth=4, max_chain_depth=6,
        dag_structure=DAGStructure.LINEAR,
        distractor_level=DISTRACTOR_LEVELS[distractor_ratio],
        transform_density=TransformDensity.STATIC,
        query_types=[QueryType.POSITION, QueryType.DISTANCE],
        num_queries=3, num_splits=3,
        seed=seed,
    )
    gen = ScenarioGenerator(cfg)
    scenario = gen.generate(f"selective_r{distractor_ratio}_s{seed}")

    response = llm.prompt(scenario.prompt)
    scores = score_response(response, scenario)
    avg = float(np.mean([s.score for s in scores]))

    for s in scores:
        kbench.assertions.assert_true(
            s.score > 0,
            expectation=(
                f"{s.query_id} (distractors={distractor_ratio}:1): "
                f"model should ignore irrelevant points. Tier={s.tier.name}"
            )
        )
    return avg

In [ ]:
eval_data = pd.DataFrame([
    {"distractor_ratio": ratio, "seed": seed}
    for ratio in [0, 1, 3, 10]
    for seed in range(5)
])

print(f"Evaluation matrix: {len(eval_data)} scenarios")
print(f"Distractor ratios: {sorted(eval_data.distractor_ratio.unique())}")

In [ ]:
@kbench.task(name="zelusbench_selective_attention")
def zelusbench_selective_attention(llm) -> tuple[float, float]:
    """Selective attention: accuracy vs distractor density.

    The 'distractor tax' is the accuracy drop from clean to noisy.
    Returns (mean_accuracy, std_dev).
    """
    with kbench.client.enable_cache():
        runs = selective_scenario.evaluate(
            llm=[llm],
            evaluation_data=eval_data,
            n_jobs=2,
            timeout=180,
            remove_run_files=True,
            stop_condition=lambda r: len(r) == len(eval_data),
            max_attempts=60,
            retry_delay=10,
        )

    results_df = runs.as_dataframe()
    scores = results_df["result"].dropna().tolist()
    accuracy = float(np.mean(scores)) if scores else 0.0
    std = float(np.std(scores)) if scores else 0.0

    kbench.assertions.assert_true(
        len(scores) > 0,
        expectation="At least some scenarios should produce results"
    )
    return accuracy, std

In [ ]:
run = zelusbench_selective_attention.run(kbench.llm)
run

In [ ]:
%choose zelusbench_selective_attention